In [41]:
from pathlib import Path
import pandas as pd
import pybedtools
from tqdm import tqdm
from collections import defaultdict

In [79]:
ROOT = Path().cwd()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
GENCODE = DATA / "gencode"
PREPROCESSING = Path("../preprocessing")
RESULTS = Path("../results")
HIMORNA_INDEX_BED = PREPROCESSING / "himorna_index-bed"
RNACHROM_INDEX_BED = PREPROCESSING / "rnachrom_index-bed"
PERMUTATION_RES = PREPROCESSING / "permutation_res"
PERMUTATION_RES_AFTER_FDR = RESULTS / "rnachrom_himorna-intersection"

HIMORNA_INDEX_BED_V2 = PREPROCESSING / "himorna_index-bed-v2"
RNACHROM_INDEX_BED_V2 = PREPROCESSING / "rnachrom_index-bed-v2"
PERMUTATION_RES_V2 = PREPROCESSING / "permutation_res-v2"
PERMUTATION_RES_AFTER_FDR_V2 = RESULTS / "rnachrom_himorna-intersection-v2"
DOCKER_R = Path("../docker_r")

In [3]:
[i for i in RNACHROM_INDEX_BED_V2.glob("*/*ENSG00000282057*")][2]#

PosixPath('../preprocessing/rnachrom_index-bed-v2/Red_C-fibroblasts-9/Red_C-fibroblasts-9-ENSG00000282057-sort.bed')

In [5]:
[pd.read_csv(i) for i in (RNACHROM_INDEX_BED_V2 / "GRID-MDA_MB_231__HTB_26_ATCC_-8").glob("GRID-MDA_MB_231__HTB_26_ATCC_-8-ENSG00000282057-sort.bed")]

[           chr1\t603522\t603544\t.\t.\t-
 0          chr1\t719426\t719448\t.\t.\t-
 1        chr1\t1260823\t1260846\t.\t.\t-
 2        chr1\t1303789\t1303811\t.\t.\t-
 3        chr1\t1547431\t1547454\t.\t.\t-
 4        chr1\t1654565\t1654587\t.\t.\t-
 ...                                  ...
 17849  chrY\t14074719\t14074742\t.\t.\t-
 17850  chrY\t18239734\t18239757\t.\t.\t-
 17851  chrY\t19537335\t19537358\t.\t.\t-
 17852  chrY\t23262864\t23262886\t.\t.\t-
 17853  chrY\t25567471\t25567493\t.\t.\t-
 
 [17854 rows x 1 columns]]

In [7]:
res_a = [pd.read_csv(i, sep="\t") for i in (RNACHROM_INDEX_BED_V2 / "GRID-MDA_MB_231__HTB_26_ATCC_-8").glob("GRID-MDA_MB_231__HTB_26_ATCC_-8-ENSG00000282057-sort.bed")][0]
res_a

,chr1,603522,603544,.,..1,-
0,chr1,719426,719448,.,.,-
1,chr1,1260823,1260846,.,.,-
2,chr1,1303789,1303811,.,.,-
3,chr1,1547431,1547454,.,.,-
4,chr1,1654565,1654587,.,.,-
...,...,...,...,...,...,...
17849,chrY,14074719,14074742,.,.,-
17850,chrY,18239734,18239757,.,.,-
17851,chrY,19537335,19537358,.,.,-
17852,chrY,23262864,23262886,.,.,-


In [10]:
to_rnachrom_example = (RNACHROM_INDEX_BED_V2 / "GRID-MDA_MB_231__HTB_26_ATCC_-8/GRID-MDA_MB_231__HTB_26_ATCC_-8-ENSG00000282057-sort.bed").absolute()
to_rnachrom_example

PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/rnachrom_index-bed-v2/GRID-MDA_MB_231__HTB_26_ATCC_-8/GRID-MDA_MB_231__HTB_26_ATCC_-8-ENSG00000282057-sort.bed')

In [11]:
res_b = [pd.read_csv(i, sep="\t") for i in (HIMORNA_INDEX_BED_V2 / "H3K9ac").glob("H3K9ac-ENSG00000282057-sort.bed")][0]

In [12]:
to_himorna_example = (HIMORNA_INDEX_BED_V2 / "H3K9ac/H3K9ac-ENSG00000282057-sort.bed").absolute()
to_himorna_example

PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H3K9ac/H3K9ac-ENSG00000282057-sort.bed')

In [13]:
res_b

,chr1,1354819,1362691
0,chr1,1419332,1419526
1,chr1,1829123,1829724
2,chr1,1843134,1843927
3,chr1,1861196,1861978
4,chr1,2211832,2215616
...,...,...,...
8930,chrX,153881655,153882030
8931,chrX,154039417,154044098
8932,chrX,154045623,154047638
8933,chrX,154493122,154493355


In [14]:
#создание файла для распараллеливания
[i for i in RNACHROM_INDEX_BED_V2.glob("*/*ENSG00000282057*")]

[PosixPath('../preprocessing/rnachrom_index-bed-v2/iMARGI-HFFc6-1/iMARGI-HFFc6-1-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/rnachrom_index-bed-v2/GRID-MDA_MB_231__HTB_26_ATCC_-8/GRID-MDA_MB_231__HTB_26_ATCC_-8-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/rnachrom_index-bed-v2/Red_C-fibroblasts-9/Red_C-fibroblasts-9-ENSG00000282057-sort.bed')]

In [15]:
[i for i in (HIMORNA_INDEX_BED_V2).glob("*/*-ENSG00000282057-sort.bed")]

[PosixPath('../preprocessing/himorna_index-bed-v2/H3K27ac/H3K27ac-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K27me3/H3K27me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K36me3/H3K36me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K4me1/H3K4me1-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K4me2/H3K4me2-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K4me3/H3K4me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K79me2/H3K79me2-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K9ac/H3K9ac-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H3K9me3/H3K9me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/H4K20me1/H4K20me1-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed-v2/meth

In [30]:
unique_lncrna_rnachrom = set([i.absolute().stem.split("-")[3] for i in RNACHROM_INDEX_BED_V2.glob("*/*") if i.stem != ".ipynb_checkpoints"])

In [31]:
len(unique_lncrna_rnachrom)

3874

In [34]:
unique_lncrna_himorna = set([i.absolute().stem.split("-")[1] for i in HIMORNA_INDEX_BED_V2.glob("*/*") if i.stem != ".ipynb_checkpoints"])

In [35]:
len(unique_lncrna_himorna)

4244

In [37]:
unique_lncrna = unique_lncrna_rnachrom & unique_lncrna_himorna
len(unique_lncrna)

3874

In [38]:
output_tsv = DOCKER_R / "pairs.tsv"

with output_tsv.open("w") as out:
    for lncrna in tqdm(unique_lncrna):
        paths_to_rnachrom_data = [i.absolute() for i in RNACHROM_INDEX_BED_V2.glob(f"*/*-{lncrna}-sort.bed")]
        paths_to_himorna_data = [i.absolute() for i in HIMORNA_INDEX_BED_V2.glob(f"*/*-{lncrna}-sort.bed")]
        
        for lncA in paths_to_rnachrom_data:
            for lncB in paths_to_himorna_data:
                out.write(f"{lncA}\t{lncB}\n")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3874/3874 [03:06<00:00, 20.74it/s]


In [39]:
paths_to_himorna_data

[PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H3K27ac/H3K27ac-ENSG00000274460-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H3K36me3/H3K36me3-ENSG00000274460-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H3K4me1/H3K4me1-ENSG00000274460-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H3K4me2/H3K4me2-ENSG00000274460-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../preprocessing/himorna_index-bed-v2/H4K20me1/H4K20me1-ENSG00000274460-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts_carefully/../p

In [ ]:
#Запуск скрипта work/git_projects/fantom.deg-himorna.peaks/docker_r/permutation-regioner-03_07_25.R через parallel

cat pairs.tsv | parallel --colsep '\t' ./permutation-regioner-03_07_25.R {1} {2} 1

In [43]:
from statsmodels.stats.multitest import multipletests

In [44]:
to_pairs_tsv = DOCKER_R / "pairs.tsv"
pair_tsv = pd.read_csv(to_pairs_tsv, sep="\t", header=None)
pair_tsv.head(2)

,0,1
0,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...
1,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...


In [46]:
Path(pair_tsv.loc[0,1]).name.split("-sort.bed")[0]

'H3K4me2-ENSG00000272256'

In [47]:
Path(pair_tsv.loc[0,1]).name.split("-sort.bed")[0]

'H3K4me2-ENSG00000272256'

In [ ]:
GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000093100-sort__vs__H3K4me2-ENSG00000093100-sort.summary.txt

In [48]:
pair_tsv["hm-lncrna"] = pair_tsv[1].apply(lambda x: Path(x).name.split("-sort.bed")[0])
pair_tsv["rnachrom_exp-lncrna"] = pair_tsv[0].apply(lambda x: Path(x).name.split("-sort.bed")[0])

In [49]:
pair_tsv["path_to_permutation_res"] = pair_tsv["rnachrom_exp-lncrna"] + "-sort__vs__" + pair_tsv["hm-lncrna"] + "-sort.summary.txt"

In [50]:
pair_tsv.loc[0, "path_to_permutation_res"]

'iMARGI-HFFc6-1-ENSG00000272256-sort__vs__H3K4me2-ENSG00000272256-sort.summary.txt'

In [51]:
pair_tsv

,0,1,hm-lncrna,rnachrom_exp-lncrna,path_to_permutation_res
0,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K4me2-ENSG00000272256,iMARGI-HFFc6-1-ENSG00000272256,iMARGI-HFFc6-1-ENSG00000272256-sort__vs__H3K4m...
1,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,methylation-ENSG00000272256,iMARGI-HFFc6-1-ENSG00000272256,iMARGI-HFFc6-1-ENSG00000272256-sort__vs__methy...
2,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K4me2-ENSG00000272256,Red_C-fibroblasts-9-ENSG00000272256,Red_C-fibroblasts-9-ENSG00000272256-sort__vs__...
3,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,methylation-ENSG00000272256,Red_C-fibroblasts-9-ENSG00000272256,Red_C-fibroblasts-9-ENSG00000272256-sort__vs__...
4,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K4me2-ENSG00000272256,iMARGI-HEK293T-2-ENSG00000272256,iMARGI-HEK293T-2-ENSG00000272256-sort__vs__H3K...
...,...,...,...,...,...
103623,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K36me3-ENSG00000274460,Red_C-K562-10-ENSG00000274460,Red_C-K562-10-ENSG00000274460-sort__vs__H3K36m...
103624,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K4me1-ENSG00000274460,Red_C-K562-10-ENSG00000274460,Red_C-K562-10-ENSG00000274460-sort__vs__H3K4me...
103625,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K4me2-ENSG00000274460,Red_C-K562-10-ENSG00000274460,Red_C-K562-10-ENSG00000274460-sort__vs__H3K4me...
103626,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H4K20me1-ENSG00000274460,Red_C-K562-10-ENSG00000274460,Red_C-K562-10-ENSG00000274460-sort__vs__H4K20m...


In [52]:
def create_df_with_pval(path_to_res):
    lnc_rna = path_to_res.stem.split("__vs__")[1].split("-")[1]
    hm = path_to_res.stem.split("__vs__")[1].split("-")[0]
    p_value = pd.read_csv(path_to_res, sep="\t")["pvalue"][0]

    rnachrom_expr = "-".join(path_to_res.stem.split("__vs__")[0].split("-sort")[0].split("-")[0:-1])
    df = pd.DataFrame({"lnc_ens":[lnc_rna], "hm":[hm], "p_value":[p_value], "rnachrom_expr":[rnachrom_expr]})
    return df

In [56]:
create_df_with_pval(PERMUTATION_RES_V2 / 'iMARGI-HFFc6-1-ENSG00000272256-sort__vs__H3K4me2-ENSG00000272256-sort.summary.txt')

,lnc_ens,hm,p_value,rnachrom_expr
0,ENSG00000272256,H3K4me2,1,iMARGI-HFFc6-1


In [58]:
list_of_res = []
for hm_lncrna, subdf in tqdm(pair_tsv.groupby(by="hm-lncrna")):
    print(hm_lncrna)
    hm_lncrna_perm_res_concat = pd.concat([create_df_with_pval(PERMUTATION_RES_V2 / i) for i in subdf["path_to_permutation_res"]])
    
    # Применим поправку на множественные сравнения к колонке 'p_value'
    corrected = multipletests(hm_lncrna_perm_res_concat["p_value"], method="fdr_bh")
    # Добавим скорректированные p-значения в DataFrame
    hm_lncrna_perm_res_concat["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values
    # При необходимости можно добавить столбец со значимостью
    hm_lncrna_perm_res_concat["significant"] = corrected[0]  # первая позиция — булевый массив значимости
    list_of_res.append(hm_lncrna_perm_res_concat)

  0%|▏                                                                                                                                                                                                                                       | 8/13926 [00:00<06:27, 35.90it/s]

H3K27ac-ENSG00000013523
H3K27ac-ENSG00000071127
H3K27ac-ENSG00000099869
H3K27ac-ENSG00000103472
H3K27ac-ENSG00000117242
H3K27ac-ENSG00000124835
H3K27ac-ENSG00000125804
H3K27ac-ENSG00000130600
H3K27ac-ENSG00000132204


  0%|▎                                                                                                                                                                                                                                      | 16/13926 [00:00<06:47, 34.14it/s]

H3K27ac-ENSG00000145075
H3K27ac-ENSG00000146147
H3K27ac-ENSG00000151303
H3K27ac-ENSG00000152931
H3K27ac-ENSG00000153113
H3K27ac-ENSG00000157306
H3K27ac-ENSG00000163364
H3K27ac-ENSG00000163597


  0%|▎                                                                                                                                                                                                                                      | 20/13926 [00:00<06:50, 33.90it/s]

H3K27ac-ENSG00000167912
H3K27ac-ENSG00000170161
H3K27ac-ENSG00000171889
H3K27ac-ENSG00000172965
H3K27ac-ENSG00000173862
H3K27ac-ENSG00000174171
H3K27ac-ENSG00000174403


... (4,310 outputs left out)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 13919/13926 [08:07<00:00, 35.94it/s]

methylation-ENSG00000286423
methylation-ENSG00000287885
methylation-ENSG00000288731
methylation-ENSG00000288985
methylation-ENSG00000290394
methylation-ENSG00000290718


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13926/13926 [08:08<00:00, 28.53it/s]

methylation-ENSG00000290890
methylation-ENSG00000290928
methylation-ENSG00000291123
methylation-ENSG00000296790
methylation-ENSG00000307585


In [121]:
all_perm_res_adj = pd.concat(list_of_res)

In [122]:
all_perm_res_adj

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000013523,H3K27ac,0.881188,iMARGI-HFFc6-1,0.881188,False
0,ENSG00000013523,H3K27ac,0.128713,GRID-MDA_MB_231__HTB_26_ATCC_-8,0.757426,False
0,ENSG00000013523,H3K27ac,0.871287,Red_C-fibroblasts-9,0.881188,False
0,ENSG00000013523,H3K27ac,0.455446,iMARGI-HEK293T-2,0.757426,False
0,ENSG00000013523,H3K27ac,0.425743,iMARGI-HUVEC-3,0.757426,False
...,...,...,...,...,...,...
0,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-3,1.000000,False
0,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-4,1.000000,False
0,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-5,1.000000,False
0,ENSG00000307585,methylation,1.000000,GRID-MM_1S-7,1.000000,False


In [123]:
all_perm_res_adj.query("lnc_ens == 'ENSG00000251661' & hm == 'H3K27ac' & rnachrom_expr == 'iMARGI-HFFc6-1'")

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000251661,H3K27ac,0.009901,iMARGI-HFFc6-1,0.01273,True


In [124]:
all_perm_res_adj["p_value"] = all_perm_res_adj["p_value"].astype(float)

In [125]:
(all_perm_res_adj["p_value"] <= 1).sum()

103628

In [126]:
all_perm_res_adj.head(3)

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000013523,H3K27ac,0.881188,iMARGI-HFFc6-1,0.881188,False
0,ENSG00000013523,H3K27ac,0.128713,GRID-MDA_MB_231__HTB_26_ATCC_-8,0.757426,False
0,ENSG00000013523,H3K27ac,0.871287,Red_C-fibroblasts-9,0.881188,False


In [127]:
len(all_perm_res_adj)

103628

In [128]:
to_file = GENCODE / 'gencode_v35-to-v47-genes-hg38-with_names-all_info.tsv'
gencode_v35_to_v47_genes_hg38_with_names = pd.read_csv(to_file, sep="\t")
gencode_v35_to_v47_genes_hg38_with_names.head(3)

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972,DDX11L1
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232,WASH7P
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267,MIR6859-1


In [129]:
set(all_perm_res_adj["lnc_ens"].unique()) - set(gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"].unique())

set()

In [130]:
all_perm_res_adj = pd.merge(left=all_perm_res_adj, right=gencode_v35_to_v47_genes_hg38_with_names[["new-ens_gene","new-gene_name"]].drop_duplicates(), how="inner", left_on="lnc_ens", right_on="new-ens_gene")
del all_perm_res_adj["new-ens_gene"]
all_perm_res_adj = all_perm_res_adj.rename(columns={"new-gene_name":"lnc"})
all_perm_res_adj

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant,lnc
0,ENSG00000013523,H3K27ac,0.881188,iMARGI-HFFc6-1,0.881188,False,ANGEL1
1,ENSG00000013523,H3K27ac,0.128713,GRID-MDA_MB_231__HTB_26_ATCC_-8,0.757426,False,ANGEL1
2,ENSG00000013523,H3K27ac,0.871287,Red_C-fibroblasts-9,0.881188,False,ANGEL1
3,ENSG00000013523,H3K27ac,0.455446,iMARGI-HEK293T-2,0.757426,False,ANGEL1
4,ENSG00000013523,H3K27ac,0.425743,iMARGI-HUVEC-3,0.757426,False,ANGEL1
...,...,...,...,...,...,...,...
103623,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-3,1.000000,False,ENSG00000307585
103624,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-4,1.000000,False,ENSG00000307585
103625,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-5,1.000000,False,ENSG00000307585
103626,ENSG00000307585,methylation,1.000000,GRID-MM_1S-7,1.000000,False,ENSG00000307585


In [131]:
all_perm_res_adj = all_perm_res_adj.rename(columns={"new-gene_name":"gene_name"})

In [133]:
all_perm_res_adj

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant,lnc
0,ENSG00000013523,H3K27ac,0.881188,iMARGI-HFFc6-1,0.881188,False,ANGEL1
1,ENSG00000013523,H3K27ac,0.128713,GRID-MDA_MB_231__HTB_26_ATCC_-8,0.757426,False,ANGEL1
2,ENSG00000013523,H3K27ac,0.871287,Red_C-fibroblasts-9,0.881188,False,ANGEL1
3,ENSG00000013523,H3K27ac,0.455446,iMARGI-HEK293T-2,0.757426,False,ANGEL1
4,ENSG00000013523,H3K27ac,0.425743,iMARGI-HUVEC-3,0.757426,False,ANGEL1
...,...,...,...,...,...,...,...
103623,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-3,1.000000,False,ENSG00000307585
103624,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-4,1.000000,False,ENSG00000307585
103625,ENSG00000307585,methylation,1.000000,iMARGI-HUVEC-5,1.000000,False,ENSG00000307585
103626,ENSG00000307585,methylation,1.000000,GRID-MM_1S-7,1.000000,False,ENSG00000307585


In [141]:
all_perm_res_adj[["lnc_ens", "hm"]].nunique()

lnc_ens    3874
hm           11
dtype: int64

In [142]:
all_perm_res_adj["lnc_ens-hm"] = all_perm_res_adj["lnc_ens"] + all_perm_res_adj["hm"]

In [143]:
all_perm_res_adj["lnc_ens-hm"].nunique()

13926

In [134]:
all_perm_res_adj["lnc_ens"].nunique(), all_perm_res_adj["lnc"].nunique()

(3874, 3874)

In [135]:
to_save = PERMUTATION_RES_AFTER_FDR_V2 / "permutation_fdr_concat.tsv"
all_perm_res_adj.to_csv(to_save, sep="\t", index=False)

In [140]:
mask = all_perm_res_adj["p_adj"] <= 0.05
all_perm_res_adj_signif = all_perm_res_adj[mask]
all_perm_res_adj_signif

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant,lnc
56,ENSG00000130600,H3K27ac,0.009901,iMARGI-HEK293T-2,0.026403,True,H19
57,ENSG00000130600,H3K27ac,0.019802,iMARGI-HUVEC-3,0.039604,True,H19
58,ENSG00000130600,H3K27ac,0.009901,iMARGI-HUVEC-4,0.026403,True,H19
59,ENSG00000130600,H3K27ac,0.009901,iMARGI-HUVEC-5,0.026403,True,H19
87,ENSG00000151303,H3K27ac,0.019802,iMARGI-HFFc6-1,0.029703,True,ENSG00000151303
...,...,...,...,...,...,...,...
103477,ENSG00000282057,methylation,0.009901,Red_C-fibroblasts-9,0.009901,True,ENSG00000282057
103509,ENSG00000282851,methylation,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_-8,0.012376,True,BISPR
103511,ENSG00000282851,methylation,0.009901,iMARGI-HUVEC-3,0.012376,True,BISPR
103512,ENSG00000282851,methylation,0.009901,iMARGI-HUVEC-4,0.012376,True,BISPR


In [138]:
all_perm_res_adj_signif["lnc_ens"].nunique(), all_perm_res_adj_signif["lnc"].nunique()

(1062, 1062)

In [139]:
to_save = PERMUTATION_RES_AFTER_FDR_V2 / "permutation_fdr_concat-signif.tsv"
all_perm_res_adj_signif.to_csv(to_save, sep="\t", index=False)

In [10]:
list_of_df = defaultdict(list)
for i in PERMUTATION_RES.glob("*.txt*"):
    lnc_rna = i.stem.split("__vs__")[1].split("-")[1]
    hm = i.stem.split("__vs__")[1].split("-")[0]
    p_value = pd.read_csv(i, sep="\t")["pvalue"][0]

    rnachrom_expr = "-".join(i.stem.split("__vs__")[0].split("-sort")[0].split("-")[0:-1])
    df = pd.DataFrame({"lnc_rna":[lnc_rna], "hm":[hm], "p_value":[p_value], "rnachrom_expr":[rnachrom_expr]})



    # Применим поправку на множественные сравнения к колонке 'p_value'
    corrected = multipletests(df["p_value"], method="fdr_bh")
    
    # Добавим скорректированные p-значения в DataFrame
    df["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values
    
    # При необходимости можно добавить столбец со значимостью
    df["significant"] = corrected[0]  # первая позиция — булевый массив значимости
    
    list_of_df[rnachrom_expr].append(df)

In [11]:
len(list_of_df)

8

In [12]:
for i in list_of_df.keys():
    list_of_df[i] = pd.concat(list_of_df[i])

In [14]:
list_of_df.keys()

dict_keys(['GRID-MDA_MB_231__HTB_26_ATCC_', 'iMARGI-HEK293T', 'iMARGI-HUVEC', 'HFFc6', 'iMARGI-HFFc6', 'GRID-MM_1S', 'Red_C-K562', 'Red_C-fibroblasts'])

In [18]:
list_of_df["Red_C-fibroblasts"]

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000267838,H3K27ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000267838,H3K9ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000267838,methylation,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000093100,H3K4me2,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000093100,H3K79me2,0.940594,Red_C-fibroblasts,0.940594,False
...,...,...,...,...,...,...
0,ENSG00000236333,methylation,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000236333,H4K20me1,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000260830,H3K9ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000260830,methylation,1.000000,Red_C-fibroblasts,1.000000,False


In [245]:
for i in list_of_df.keys():
    a = list_of_df[i][list_of_df[i]["significant"] == "True"]
    break

In [246]:
a

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant


In [206]:
res_df = pd.concat(list_of_df)

In [207]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Применим поправку на множественные сравнения к колонке 'p_value'
corrected = multipletests(res_df["p_value"], method="fdr_bh")

# Добавим скорректированные p-значения в DataFrame
res_df["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values

# При необходимости можно добавить столбец со значимостью
res_df["significant"] = corrected[0]  # первая позиция — булевый массив значимости

In [208]:
res_df

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000282057,H3K9ac,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057,0.242144,False
0,ENSG00000255328,H3K27ac,0.950495,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HUVEC-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,HFFc6-ENSG00000255328,1.000000,False
...,...,...,...,...,...,...
0,ENSG00000265369,methylation,1.000000,HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,methylation,1.000000,iMARGI-HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HEK293T-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HUVEC-ENSG00000265369,1.000000,False


In [209]:
res_df

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000282057,H3K9ac,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057,0.242144,False
0,ENSG00000255328,H3K27ac,0.950495,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HUVEC-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,HFFc6-ENSG00000255328,1.000000,False
...,...,...,...,...,...,...
0,ENSG00000265369,methylation,1.000000,HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,methylation,1.000000,iMARGI-HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HEK293T-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HUVEC-ENSG00000265369,1.000000,False


In [210]:
res_df["p_adj"].min()

0.24214377959535077

In [148]:
i.stem.split("__vs__")[1].split("-")[0]

'H3K9ac'

In [149]:
i.stem.split("__vs__")[1].split("-")[1]

'ENSG00000282057'

In [151]:
i.stem.split("__vs__")[0].split("-sort")[0]

'GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057'